## 4. Preparación del conjunto de datos

### Integración de datos

Vamos a juntar las tres fuentes de datos en una unica. Vamos a usar los dataset de edicios, carreteras y arbolados para calcular una serie de variables para añadir a nuestro dataset. 

Estas variables son: Distancia a carreteras, cantidad de arboles en un radio, altura media de edificios en un rango. 

Podemos considerar que son datos que influyen en la temperatura, ya que estar cerca de una carretera como la S30, tener un parque con muchos arboles cerca o edificios de gran altura; pueden influir claramente en la temperatura superficial. 

In [1]:
import os
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
import warnings
import geopandas as gpd

In [2]:
sevilla_dataset_pixels_csv_path = os.path.join("..", "datasets","raw", "sevilla_dataset_pixels.csv")
arbolado_csv_path = os.path.join("..", "datasets","raw", "arbolado_sevilla.csv")
carreteras_csv_path = os.path.join("..", "datasets","raw", "carreteras_sevilla.csv")
edificios_csv_path = os.path.join("..", "datasets","raw", "edificios_sevilla.csv")

processed_dir = os.path.join("..", "datasets", "processed")
os.makedirs(processed_dir, exist_ok=True)
output_file = os.path.join(processed_dir, "sevilla_dataset_preparado.csv")

In [3]:
for ruta in (sevilla_dataset_pixels_csv_path, arbolado_csv_path, carreteras_csv_path, edificios_csv_path):
    if not os.path.exists(ruta):
        raise FileNotFoundError(f"No existe el fichero requerido: {ruta}")

df_sevilla_pixeles = pd.read_csv(sevilla_dataset_pixels_csv_path)
df_arboles = pd.read_csv(arbolado_csv_path)
df_carreteras = pd.read_csv(carreteras_csv_path)
df_edificios = pd.read_csv(edificios_csv_path)



In [4]:
# 2. Proyección local a metros (Aproximación para Sevilla)
# 1 grado de Latitud = ~111320 metros.
# 1 grado de Longitud en Sevilla (Lat 37.4) = Cos(37.4) * 111320 =~ 88400 metros.
FACTOR_LON = 88400
FACTOR_LAT = 111320
# Convertimos los píxeles principales a metros
pixels_m = np.column_stack((
    df_sevilla_pixeles['Longitude'].values * FACTOR_LON,
    df_sevilla_pixeles['Latitude'].values * FACTOR_LAT
))


## Carreteras
### Respecto a las carreteras, no vamos a realizar ninguna modificacion de dataset. Vamos a calcular dos tipos de vias, de alta capacidad y las demas, y vamos a crear una columna para cada via que nos indique la distancia a la via

In [8]:
# --- 3. MATCH DE CARRETERAS (Jerarquía de Tráfico) ---
print("  -> 🚗 Calculando exposición al tráfico por jerarquía...")

# Separamos vías de alta capacidad de vías puramente urbanas
df_vias_rapidas = df_carreteras[df_carreteras['tipo'].isin(['motorway', 'trunk'])]
df_vias_urbanas = df_carreteras[~df_carreteras['tipo'].isin(['motorway', 'trunk'])]

# Árbol 1: Vías rápidas (SE-30, autovías)
rapidas_m = np.column_stack((df_vias_rapidas['longitud'] * FACTOR_LON, df_vias_rapidas['latitud'] * FACTOR_LAT))
tree_rapidas = cKDTree(rapidas_m)
dist_rapidas, _ = tree_rapidas.query(pixels_m, k=1)
df_sevilla_pixeles['D2R_HighCapacity_m'] = dist_rapidas




  -> 🚗 Calculando exposición al tráfico por jerarquía...


In [9]:
# Árbol 2: Vías urbanas (primary, secondary)
urbanas_m = np.column_stack((df_vias_urbanas['longitud'] * FACTOR_LON, df_vias_urbanas['latitud'] * FACTOR_LAT))
tree_urbanas = cKDTree(urbanas_m)
dist_urbanas, _ = tree_urbanas.query(pixels_m, k=1)
df_sevilla_pixeles['D2R_Urban_m'] = dist_urbanas



## Arbolado

### En cuanto al arbolado, de momento solo vamos a calcular cuantos arboles existen en un radio de 50m para cada punto. Se podria plantear introducir que tipo de arboles de alguna forma ya que no todos los arboles dan la misma sombra.

In [15]:
# --- 4. MATCH DE ARBOLADO (Densidad en 50m) ---
print("  -> 🌳 Calculando densidad de biomasa (Radio 50m)...")
arboles_m = np.column_stack((df_arboles['longitud'] * FACTOR_LON, df_arboles['latitud'] * FACTOR_LAT))
tree_arboles = cKDTree(arboles_m)
indices_arboles = tree_arboles.query_ball_point(pixels_m, r=50)
df_sevilla_pixeles['Tree_Density_50m'] = [len(idx) for idx in indices_arboles]



  -> 🌳 Calculando densidad de biomasa (Radio 50m)...


## Edificios
### Ya pudimos observar cuando hicimos el analisis, que habia errores en las alturas maximas habiaendo outliers. Vamos a eliminarlos antes de integrar el dataset.

Sabiendo que el edificio más alto de Sevilla es Torre Sevilla, y mide 180 metros, vamos a eliminar todo lo superior a 200m. 
Estos valores tan altos rompian la media y la desviación estandar. 

In [5]:
# Mantenemos todos los registros cuya altura sea menor o igual a 200
df_edificios_limpio = df_edificios[df_edificios['altura_estimada'] <= 200.0]

# Comprobación rápida del filtrado
filas_antes = len(df_edificios)
filas_despues = len(df_edificios_limpio)
eliminados = filas_antes - filas_despues

print(f"Se han eliminado {eliminados} registros con altura superior a 200 metros.")

Se han eliminado 3 registros con altura superior a 200 metros.


In [16]:
# --- 5. MATCH DE EDIFICIOS (Morfología y Proxy SVF en 100m) ---
print("  -> 🏢 Calculando morfología urbana y Sky View Factor (Radio 100m)...")
edificios_m = np.column_stack((df_edificios['longitud'] * FACTOR_LON, df_edificios['latitud'] * FACTOR_LAT))
tree_edificios = cKDTree(edificios_m)

# Buscamos todos los edificios en un radio de 100 metros del píxel
indices_edificios = tree_edificios.query_ball_point(pixels_m, r=100)
alturas_array = df_edificios['altura_estimada'].values

# Cálculos vectorizados para evitar bucles lentos
# 1. Cantidad de edificios (Densidad constructiva)
df_sevilla_pixeles['Building_Density_100m'] = [len(idx) for idx in indices_edificios]

# 2. Altura media (Si no hay edificios, la altura es 0)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    # np.mean lanza warning si la lista está vacía, lo capturamos e inyectamos 0
    df_sevilla_pixeles['Avg_Building_Height_100m'] = [
        np.mean(alturas_array[idx]) if len(idx) > 0 else 0.0 
        for idx in indices_edificios
    ]


  -> 🏢 Calculando morfología urbana y Sky View Factor (Radio 100m)...


In [17]:
df_sevilla_pixeles.to_csv(output_file, index=False, encoding='utf-8')

## Dataset extras

### Dado que tenemos una cantidad muy alta de puntos, esto supone un problema para el renderizado en una web tal y como queremos hacer, ahora vamos a realizar otro csv, una agrupacion por barrios, para la web que generaremos para la visualización de datos y prueba de modelos.

### El objetivo es tener la media de los datos agrupados por barrios, para eso nos descargamos de la web '../datasets/raw/barrios_sevilla.geojson' que es un listado de barrios y su poligono respecto a un mapa en coordenadas, para poder delimitarlos.

In [18]:
# Mostramos las primeras filas para verificar
df_sevilla_pixeles.head()

,Longitude,Latitude,NDVI,NDBI,Albedo,D2W_meters,LST_Target,D2R_HighCapacity_m,D2R_Urban_m,Tree_Density_50m,Building_Density_100m,Avg_Building_Height_100m
0,-6.029941,37.449956,0.505166,-0.178238,0.157998,20.000000,40.960874,2743.999249,2776.807410,1,0,0.0
1,-6.029762,37.449956,0.728687,-0.395285,0.181745,20.000000,40.714777,2743.879563,2760.939843,1,0,0.0
2,-6.029582,37.449956,0.763079,-0.424442,0.149877,20.000000,40.714777,2743.851804,2745.072446,1,0,0.0
3,-6.029402,37.449956,0.775685,-0.418584,0.147389,22.360680,40.379811,2743.915975,2729.205220,1,0,0.0
4,-6.029223,37.449956,0.835151,-0.530323,0.201774,28.284271,40.379811,2744.072070,2713.338171,1,0,0.0


In [ ]:
# 2. Convertimos a GeoDataFrame
gdf = gpd.GeoDataFrame(df_sevilla_pixeles, geometry=gpd.points_from_xy(df_sevilla_pixeles.Longitude, df_sevilla_pixeles.Latitude))

# 3. Cargamos los barrios de Sevilla (GeoJSON)
path_barrios = os.path.join("..", "datasets", "raw", "barrios_sevilla.geojson")
barrios = gpd.read_file(path_barrios)

# 4. Asignamos cada punto a un barrio (Spatial Join)
# Esto une los puntos con los polígonos de los barrios automáticamente
gdf_joined = gpd.sjoin(gdf, barrios, how="inner", predicate="within")

columnas_media = [
    'NDVI', 'NDBI', 'Albedo', 'D2W_meters', 'LST_Target',
    'D2R_HighCapacity_m', 'D2R_Urban_m', 'Tree_Density_50m',
    'Building_Density_100m', 'Avg_Building_Height_100m'
]

# 5. Calculamos la media de todas las variables por barrio
resumen_barrios_df = gdf_joined.groupby('name')[columnas_media].mean().reset_index()

# Hacemos un 'merge' con el GeoDataFrame original de barrios para recuperar sus políGONOS
resumen_barrios_gdf = resumen_barrios_df.merge(barrios[['name', 'geometry']], on='name')

# Convertimos a GeoDataFrame definitivo
resumen_barrios_gdf = gpd.GeoDataFrame(resumen_barrios_gdf, geometry='geometry')

/var/folders/_x/_zlpy6gs1lb0cs10grzc5k4m0000gn/T/ipykernel_37482/3341433429.py:9: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: None
Right CRS: EPSG:4326

  gdf_joined = gpd.sjoin(gdf, barrios, how="inner", predicate="within")


In [ ]:
path_mapa_barrios = os.path.join("..", "datasets", "processed", "mapa_barrios_temperatura.geojson")
resumen_barrios_gdf.to_file(path_mapa_barrios, driver='GeoJSON')


In [ ]:
path_mapa_barrios_web = os.path.join("..", "web", "frontend-sevilla", "src", "assets", "mapa_barrios_temperatura.geojson")
resumen_barrios_gdf.to_file(path_mapa_barrios_web, driver='GeoJSON')

### Dado que nuestro dataset pricipal es muy grande y contiene muchisimos puntos, vamos a dividirlo por barrios para facilitar su uso en la web, es decir el mismo dataset pero con un filtro por cada barrio. Esto lo haremos a partir del GeoDataFrame que ya tiene asignado el barrio a cada punto (gdf_joined). De esta forma, cada CSV tendrá solo los puntos de ese barrio, lo que facilitará su carga y visualización en la web sin necesidad de cargar todo el dataset completo.

In [ ]:
# 7. NUEVO: Crear un CSV por cada barrio
print("💾 Generando archivos CSV detallados por barrio...")

# Definimos las rutas de destino
path_barrios_csv = os.path.join("..", "datasets", "processed", "barrios")
path_barrios_csv_web = os.path.join("..", "web", "frontend-sevilla", "src", "assets", "barrios")
rutas_destino = [path_barrios_csv, path_barrios_csv_web]

for nombre_barrio, grupo in gdf_joined.groupby('name'):
    # Limpiamos el nombre para que sea un nombre de archivo válido (quitamos espacios y pasamos a minúsculas)
    nombre_archivo = f"detail_{nombre_barrio.replace(' ', '_').lower()}.csv"
    
    for ruta in rutas_destino:
        # Aseguramos que el directorio exista
        os.makedirs(ruta, exist_ok=True)
        
        # Guardamos el CSV filtrado para este barrio
        # (Si solo quieres ciertas columnas, puedes poner .to_csv(ruta + nombre_archivo, columns=['col1', 'col2', ...]))
        grupo.to_csv(os.path.join(ruta, nombre_archivo), index=False, encoding='utf-8')

print("✅ Todos los archivos CSV han sido generados exitosamente.")

💾 Generando archivos CSV detallados por barrio...
✅ Todos los archivos CSV han sido generados exitosamente.
